In [ ]:
# Python packages
import scanpy as sc
import scvi
import bbknn
import scib
import numpy as np
import pandas as pd  
import matplotlib.pyplot as plt
import matplotlib
import anndata as ad
# R interface
from rpy2.robjects import pandas2ri
from rpy2.robjects import r
import rpy2.rinterface_lib.callbacks
import anndata2ri

pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

In [ ]:
%%R
library(Seurat)

In [ ]:
import glob
h5ad_files = glob.glob("/Users/yanjun/Desktop/scanpy/4/*.h5ad")
adatas = [sc.read_h5ad(f) for f in h5ad_files]

In [ ]:
for adata in adatas:
    del adata.obs['leiden_res_1_5'] 
    del adata.obs['leiden_res_1'] 
    del adata.obs['leiden_res_0_6'] 
    adata.obsm = {}

In [ ]:
adatas

In [ ]:
adata = ad.concat(adatas, join='outer', index_unique=None)

In [ ]:
adata

In [ ]:
 del adatas

In [ ]:
# Normalise, HVG and PCA
adata.X = adata.layers["counts"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata.layers["logcounts"] = adata.X.copy()

In [ ]:
# Neighbours, UMAP and clustering
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3)
sc.tl.pca(adata)
sc.pp.scale(adata)
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=50)
sc.tl.umap(adata)

In [ ]:
print((adata.var['highly_variable'] == True).sum())

In [ ]:
sc.tl.leiden(
    adata,
    resolution= 1,
    random_state=0,
    flavor="igraph",
    n_iterations=2,
    directed=False,
    key_added='uncorrected_1'
)

In [ ]:
sc.pl.umap(adata, color=['timepoint'], wspace=1)

In [ ]:
sc.pl.umap(adata, color=['pre_annotation'], size=100)

In [ ]:
sc.pl.umap(adata, color=['uncorrected_0_8', 'uncorrected_1'])

In [ ]:
# Differential expression
sc.tl.rank_genes_groups(adata, groupby="uncorrected_1", method="wilcoxon", layer = 'data')
sc.pl.rank_genes_groups_dotplot(adata, groupby="uncorrected_1", standard_scale="var", n_genes=20, dendrogram = None)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, groupby="uncorrected_1", standard_scale="var", n_genes=50, dendrogram = None)

In [ ]:
sc.pl.umap(adata, color=['LYVE1', 'condition'], size=100)

In [ ]:
sc.pl.dotplot(adata, ['C1QA','C1QC','CD68','PTPRC','S100A8','LYVE1', 'FOLR2', 'F13A1', 'SELENOP', 'VCAN',
                      'LYN', 'CD14', 'FCGR3A',
                    'MRC1', 'HLA-DRA', 'CD68', 'HLA-A', 'PSAP', 'S100A9'], 
              groupby = 'uncorrected_1', dendrogram = None, dot_max = 0.6
             )

In [ ]:
batch_key = 'timepoint'

In [ ]:
sc.pp.highly_variable_genes(
    adata, n_top_genes=5000, flavor="seurat", batch_key=batch_key
)
adata.var

In [ ]:
n_batches = adata.var["highly_variable_nbatches"].value_counts()
ax = n_batches.plot(kind="bar")
n_batches

In [ ]:
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
adata_hvg

In [ ]:
sc.pl.umap(adata_hvg, color=['timepoint'], wspace=1)

In [ ]:
del adata

In [ ]:
#### BBKNN and Harmony

In [ ]:
neighbors_within_batch = 25 if adata_hvg.n_obs > 100000 else 3
neighbors_within_batch

In [ ]:
adata_bbknn = adata_hvg.copy()
adata_bbknn.X = adata_bbknn.layers["logcounts"].copy()
sc.pp.pca(adata_bbknn)

In [ ]:
bbknn.bbknn(
    adata_bbknn, batch_key=batch_key, neighbors_within_batch=neighbors_within_batch
)
adata_bbknn

In [ ]:
sc.tl.umap(adata_bbknn)
sc.pl.umap(adata_bbknn, color=[batch_key], wspace=1)

In [ ]:
adata_harmony = adata_hvg.copy()

In [ ]:
adata_bbknn = adata_hvg.copy()
adata_harmony.X = adata_harmony.layers["logcounts"].copy()
sc.pp.pca(adata_harmony)

In [ ]:
import scanpy.external as sce
sce.pp.harmony_integrate(adata_harmony, 'timepoint')

In [ ]:
adata_harmony

In [ ]:
sc.pl.pca(adata_harmony, color = batch_key)

In [ ]:
adata_harmony.obsm['X_pca'] = adata_harmony.obsm['X_pca_harmony']
sc.pp.neighbors(adata_harmony, n_neighbors=10, n_pcs=30)
sc.tl.umap(adata_harmony)
sc.pl.umap(adata_harmony, color = batch_key)

In [ ]:
# MNN (seurat cca)

In [ ]:
adata_seurat = adata_hvg.copy()
# Convert categorical columns to strings
adata_seurat.obs[batch_key] = adata_seurat.obs[batch_key].astype(str)
# Delete uns as this can contain arbitrary objects which are difficult to convert
del adata_seurat.uns
adata_seurat

In [ ]:
%%R -i adata_seurat
adata_seurat

In [ ]:
%%R -i adata_seurat
seurat <- as.Seurat(adata_seurat, counts = "counts", data = "logcounts")
seurat

In [ ]:
%%R -i batch_key
batch_list <- SplitObject(seurat, split.by = batch_key)
batch_list

In [ ]:
%%R
anchors <- FindIntegrationAnchors(batch_list, anchor.features = rownames(seurat))
anchors

In [ ]:
%%R
integrated <- IntegrateData(anchors)
integrated

In [ ]:
%%R -o integrated_expr
# Extract the integrated expression matrix
integrated_expr <- GetAssayData(integrated)
# Make sure the rows and columns are in the same order as the original object
integrated_expr <- integrated_expr[rownames(seurat), colnames(seurat)]
# Transpose the matrix to AnnData format
integrated_expr <- t(integrated_expr)
print(integrated_expr[1:10, 1:10])

In [ ]:
adata_seurat.X = integrated_expr
adata_seurat.layers["seurat"] = integrated_expr
print(adata_seurat)

In [ ]:
# Reset the batch colours because we deleted them earlier
adata_seurat.uns[batch_key + "_colors"] = [
    "#1b9e77",
    "#d95f02",
    "#7570b3",
]
sc.tl.pca(adata_seurat)
sc.pp.neighbors(adata_seurat)
sc.tl.umap(adata_seurat)
sc.pl.umap(adata_seurat, color=[batch_key], wspace=1)

In [ ]:
adata_stacas = adata_hvg.copy()
# Convert categorical columns to strings
adata_stacas.obs[batch_key] = adata_stacas.obs[batch_key].astype(str)
# Delete uns as this can contain arbitrary objects which are difficult to convert
del adata_stacas.uns
adata_stacas

In [ ]:
%%R -i adata_stacas
stacas <- as.Seurat(adata_stacas, counts = "counts", data = "logcounts")
stacas

In [ ]:
%%R 
library(dplyr)
library(STACAS)
object_integrated <- stacas %>% SplitObject(split.by = "timepoint") %>%
      Run.STACAS(dims = 1:30, anchor.features = rownames(stacas)) %>%
      RunUMAP(dims = 1:30) 

In [ ]:
%%R -o integrated_expr
# Extract the integrated expression matrix
integrated_expr <- GetAssayData(object_integrated)
# Make sure the rows and columns are in the same order as the original object
# Transpose the matrix to AnnData format
integrated_expr <- t(integrated_expr)
print(integrated_expr[1:10, 1:10])

In [ ]:
%%R
object_integrated

In [ ]:
adata_stacas.X = integrated_expr
adata_stacas.layers["stacas"] = integrated_expr
print(adata_stacas)

In [ ]:
%%R -o umap_embeddings
umap_embeddings <- object_integrated@reductions$umap@cell.embeddings

In [ ]:
import numpy as np

In [ ]:
%%R -i umap_embeddings
adata_stacas.obsm['X_umap'] = np.array(umap_embeddings)

In [ ]:
sc.pl.umap(adata_stacas, color = batch_key)

In [ ]:
import pandas as pd

# List of your AnnData objects
adatas = [adata_hvg, adata_bbknn, adata_harmony, adata_seurat, adata_stacas]

# Initialize a dictionary to store scores
scores = {}
for adata in adatas:
    score = scib.metrics.ilisi_graph(adata, batch_key=batch_key, type_='full').mean() 
    scores['adata'] = score

# Convert the scores dictionary to a DataFrame
scores_df = pd.DataFrame.from_dict(scores, orient='index', columns=['iLISI Score'])



In [ ]:
adata_harmony

In [ ]:
hvg_scib = scib.me.ilisi_graph(adata_hvg, batch_key=batch_key, type_)
har_scib =  scib.me.ilisi_graph(adata_harmony, batch_key=batch_key, type="embed", use_rep="X_umap")
seu_scib = scib.me.ilisi_graph(adata_seurat, batch_key=batch_key, type="full")
bbknn_scib = scib.me.ilisi_graph(adata_bbknn, batch_key=batch_key, type="full")
sta_scib = scib.me.ilisi_graph(adata_stacas, batch_key=batch_key, type="full")



In [ ]:
metrics_scvi = scib.metrics.metrics_fast(
    adata, adata_harmony, batch_key, embed="X_harmony"
)
metrics_bbknn = scib.metrics.metrics_fast(adata, adata_bbknn, batch_key)
metrics_seurat = scib.metrics.metrics_fast(adata, adata_seurat, batch_key)
metrics_hvg = scib.metrics.metrics_fast(adata, adata_hvg, batch_key)